# ViT Paper Replication â€” Colab Verification

Trains both models and prints final test accuracy.

> **Before running:** Runtime â†’ Change runtime type â†’ **T4 GPU**

In [ ]:
# Clone repo
!git clone https://github.com/Roopesh-BR/VisionTransformer-Paper-Replication.git
%cd VisionTransformer-Paper-Replication

In [ ]:
# Install dependencies
!pip install -r requirements.txt -q
!pip install -e . -q

In [ ]:
# Confirm GPU
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
import os
import zipfile
import requests
import torch
from torch import nn
from torchvision import transforms
from torchvision.models import vit_b_16, ViT_B_16_Weights
from pathlib import Path

from vit.data import create_dataloaders
from vit.model import ViT
from vit.engine import train
from vit.utils import save_model

DATA_URL = 'https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi.zip'

def download_data(source: str, destination: str, remove_source: bool = True) -> Path:
    data_path = Path('data/')
    image_path = data_path / destination
    if image_path.is_dir():
        print(f'[INFO] {image_path} already exists, skipping download.')
    else:
        image_path.mkdir(parents=True, exist_ok=True)
        target_file = Path(source).name
        with open(data_path / target_file, 'wb') as f:
            print(f'[INFO] Downloading {target_file}...')
            f.write(requests.get(source).content)
        with zipfile.ZipFile(data_path / target_file, 'r') as z:
            z.extractall(image_path)
        if remove_source:
            os.remove(data_path / target_file)
    return image_path

image_path = download_data(source=DATA_URL, destination='pizza_steak_sushi')

In [ ]:
# Train custom ViT from scratch (10 epochs)
torch.manual_seed(42)
custom_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
train_dl, test_dl, class_names = create_dataloaders(
    train_dir=image_path / 'train',
    test_dir=image_path / 'test',
    transform=custom_transform,
    batch_size=32,
)

custom_vit = ViT(num_classes=len(class_names)).to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(custom_vit.parameters(), lr=3e-3, weight_decay=0.1)

custom_results = train(
    model=custom_vit,
    train_dataloader=train_dl,
    test_dataloader=test_dl,
    optimizer=optimizer,
    loss_fn=loss_fn,
    epochs=10,
    device=device,
)
save_model(custom_vit, target_dir='models', model_name='custom_vit.pth')

In [ ]:
# Fine-tune pretrained ViT-B/16 (10 epochs)
weights = ViT_B_16_Weights.DEFAULT
pretrained_vit = vit_b_16(weights=weights)
for param in pretrained_vit.parameters():
    param.requires_grad = False
pretrained_vit.heads = nn.Linear(in_features=768, out_features=len(class_names))
pretrained_vit = pretrained_vit.to(device)

auto_transforms = weights.transforms()
train_dl_pt, test_dl_pt, _ = create_dataloaders(
    train_dir=image_path / 'train',
    test_dir=image_path / 'test',
    transform=auto_transforms,
    batch_size=32,
)
optimizer_pt = torch.optim.Adam(
    params=filter(lambda p: p.requires_grad, pretrained_vit.parameters()),
    lr=1e-3,
)

pretrained_results = train(
    model=pretrained_vit,
    train_dataloader=train_dl_pt,
    test_dataloader=test_dl_pt,
    optimizer=optimizer_pt,
    loss_fn=loss_fn,
    epochs=10,
    device=device,
)
save_model(pretrained_vit, target_dir='models', model_name='pretrained_vit.pth')

In [ ]:
print('=' * 42)
print('FINAL RESULTS')
print('=' * 42)
print(f'  Custom ViT from scratch (10ep): {custom_results["test_acc"][-1]:.1%}')
print(f'  Pretrained ViT-B/16  (10ep):   {pretrained_results["test_acc"][-1]:.1%}')
print('=' * 42)